In [1]:
%pip install google-genai python-dotenv

Defaulting to user installation because normal site-packages is not writeable
     ---------------------------------------- 0.0/58.6 kB ? eta -:--:--
     ------ --------------------------------- 10.2/58.6 kB ? eta -:--:--
     ------------------- ------------------ 30.7/58.6 kB 435.7 kB/s eta 0:00:01
     -------------------------------------- 58.6/58.6 kB 620.1 kB/s eta 0:00:00
   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   ----- ---------------------------------- 0.2/1.2 MB 5.3 MB/s eta 0:00:01
   ---------------- ----------------------- 0.5/1.2 MB 6.4 MB/s eta 0:00:01
   ------------------------------- -------- 1.0/1.2 MB 8.6 MB/s eta 0:00:01
   ---------------------------------------  1.2/1.2 MB 8.4 MB/s eta 0:00:01
   ---------------------------------------- 1.2/1.2 MB 7.0 MB/s eta 0:00:00
   ---------------------------------------- 0.0/267.9 kB ? eta -:--:--
   ---------------------------------------- 267.9/267.9 kB 8.1 MB/s eta 0:00:00
   --------------


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
from dotenv import load_dotenv
import os
from google import genai

load_dotenv()
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

for m in client.models.list():
    if "flash" in m.name:
        print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-flash-preview-tts
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/gemini-3.1-flash-tts-preview
models/gemini-3.8-flash-tts
models/gemini-3.8-flash-lite-tts
models/gemini-2.5-flash-native-audio-latest
models/gemini-2.5-flash-native-audio-preview-09-2025
models/gemini-2.5-flash-native-audio-preview-12-2025
models/gemini-3.1-flash-live-preview


In [3]:
from dotenv import load_dotenv
import os, re
import duckdb
from google import genai

load_dotenv()
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

con = duckdb.connect()
con.execute("CREATE VIEW orders AS SELECT * FROM read_csv_auto('data/dataco_clean.csv')")
schema = con.execute("DESCRIBE orders").df()[["column_name", "column_type"]].to_string(index=False)

MODEL = "gemini-flash-latest"

def make_sql(question):
    prompt = f"""You write DuckDB SQL for one table named orders.
Columns:
{schema}

Rules:
- Return only one SQL SELECT query. No explanation, no markdown.
- Put column names with spaces in double quotes.
- Late_delivery_risk: 1 = late, 0 = not late.
- For late-delivery questions, exclude rows where "Delivery Status" = 'Shipping canceled'.

Question: {question}"""
    response = client.models.generate_content(model=MODEL, contents=prompt)
    sql = response.text.strip()
    sql = re.sub(r"^```(sql)?|```$", "", sql, flags=re.IGNORECASE).strip()
    return sql

def ask(question):
    sql = make_sql(question)
    blocked = r"\b(insert|update|delete|drop|create|alter|copy|attach)\b"
    if not sql.lower().startswith(("select", "with")) or re.search(blocked, sql.lower()):
        print("Blocked unsafe query:\n", sql)
        return None
    print("SQL:\n", sql)
    return con.execute(sql).df()

ask("What is the late delivery rate for each shipping mode?")

SQL:
 SELECT "Shipping Mode", AVG(Late_delivery_risk) AS late_delivery_rate FROM orders WHERE "Delivery Status" != 'Shipping canceled' GROUP BY "Shipping Mode"


,Shipping Mode,late_delivery_rate
0,Second Class,0.798290
1,First Class,1.000000
2,Same Day,0.479285
3,Standard Class,0.397691


In [5]:
from dotenv import load_dotenv
import os, re
import duckdb
from google import genai

load_dotenv()
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

con = duckdb.connect()
con.execute("CREATE VIEW orders AS SELECT * FROM read_csv_auto('data/dataco_clean.csv')")
schema = con.execute("DESCRIBE orders").df()[["column_name", "column_type"]].to_string(index=False)

MODEL = "gemini-flash-latest"

def make_sql(question):
    prompt = f"""You write DuckDB SQL for one table named orders.
Columns:
{schema}

Rules:
- Return only one SQL SELECT query. No explanation, no markdown.
- Put column names with spaces in double quotes.
- Late_delivery_risk: 1 = late, 0 = not late.
- For late-delivery questions, exclude rows where "Delivery Status" = 'Shipping canceled'.
- "Cancelled" means "Delivery Status" = 'Shipping canceled'. Do not use "Order Status" for cancellations.

Question: {question}"""
    response = client.models.generate_content(model=MODEL, contents=prompt)
    sql = response.text.strip()
    sql = re.sub(r"^```(sql)?|```$", "", sql, flags=re.IGNORECASE).strip()
    return sql

def ask(question):
    sql = make_sql(question)
    blocked = r"\b(insert|update|delete|drop|create|alter|copy|attach)\b"
    if not sql.lower().startswith(("select", "with")) or re.search(blocked, sql.lower()):
        print("Blocked unsafe query:\n", sql)
        return None
    print("SQL:\n", sql)
    return con.execute(sql).df()

ask("How many order items were cancelled?")

SQL:
 SELECT COUNT(*) FROM orders WHERE "Delivery Status" = 'Shipping canceled';


,count_star()
0,7754


In [6]:
import time
import pandas as pd

tests = [
    ("How many order items are there in total?",
     "SELECT COUNT(*) FROM orders"),
    ("What is the overall late delivery rate?",
     """SELECT AVG(Late_delivery_risk) FROM orders WHERE "Delivery Status" <> 'Shipping canceled'"""),
    ("How many order items were cancelled?",
     """SELECT COUNT(*) FROM orders WHERE "Delivery Status" = 'Shipping canceled'"""),
    ("What is the total profit?",
     """SELECT SUM("Order Profit Per Order") FROM orders"""),
    ("What share of order items lose money?",
     """SELECT AVG(CASE WHEN "Order Profit Per Order" < 0 THEN 1 ELSE 0 END) FROM orders"""),
    ("Which region has the highest late delivery rate, and what is the rate?",
     """SELECT "Order Region", AVG(Late_delivery_risk) FROM orders WHERE "Delivery Status" <> 'Shipping canceled' GROUP BY 1 ORDER BY 2 DESC LIMIT 1"""),
    ("What is the total sales for each market?",
     """SELECT Market, SUM(Sales) FROM orders GROUP BY Market"""),
    ("What is the average profit per order item for each shipping mode?",
     """SELECT "Shipping Mode", AVG("Order Profit Per Order") FROM orders GROUP BY 1"""),
    ("How many late order items are there for each shipping mode?",
     """SELECT "Shipping Mode", SUM(Late_delivery_risk) FROM orders WHERE "Delivery Status" <> 'Shipping canceled' GROUP BY 1"""),
    ("What are the top 3 categories by total profit? Show category and total profit.",
     """SELECT "Category Name", SUM("Order Profit Per Order") FROM orders GROUP BY 1 ORDER BY 2 DESC LIMIT 3"""),
]

def to_values(df):
    vals = set()
    for v in df.to_numpy().flatten():
        try:
            vals.add(round(float(v), 2))
        except (TypeError, ValueError):
            vals.add(str(v).strip())
    return vals

results = []
for question, truth_sql in tests:
    expected = to_values(con.execute(truth_sql).df())
    try:
        got_df = ask(question)
        got = to_values(got_df) if got_df is not None else set()
        passed = expected.issubset(got)
    except Exception as e:
        print("Error:", e)
        passed = False
    results.append({"question": question, "passed": passed})
    time.sleep(6)

report = pd.DataFrame(results)
print(report.to_string())
print("Accuracy:", round(report["passed"].mean() * 100, 1), "%")

SQL:
 SELECT COUNT(*) FROM orders;
SQL:
 SELECT AVG("Late_delivery_risk") AS late_delivery_rate FROM orders WHERE "Delivery Status" != 'Shipping canceled';
SQL:
 SELECT COUNT(*) FROM orders WHERE "Delivery Status" = 'Shipping canceled';
SQL:
 SELECT SUM("Order Profit Per Order") FROM orders;
SQL:
 SELECT AVG(CASE WHEN "Order Profit Per Order" < 0 THEN 1.0 ELSE 0.0 END) AS share_losing_money FROM orders;
SQL:
 SELECT "Order Region", AVG("Late_delivery_risk") AS late_delivery_rate FROM orders WHERE "Delivery Status" != 'Shipping canceled' GROUP BY "Order Region" ORDER BY late_delivery_rate DESC LIMIT 1;
SQL:
 SELECT "Market", SUM("Sales") AS "Total Sales" FROM orders GROUP BY "Market"
SQL:
 SELECT "Shipping Mode", AVG("Order Profit Per Order") AS "Average Profit" FROM orders GROUP BY "Shipping Mode"
Error: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: ht